# Context Delay Metrics (Polars)

Reports robust delay metrics by line, direction, local hour, and weekday/weekend context using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

context_metrics = load_polars_script("polars_context_delay_metrics", "context-delay-metrics.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
LIMIT = 50
MIN_OBSERVATIONS = 50
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"
LINE_REF = None
DIRECTION_REF = None
DAY_TYPE = "all"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    line_ref = LINE_REF
    direction_ref = DIRECTION_REF
    day_type = DAY_TYPE

buckets = context_metrics.load_delay_buckets_for_args(Args)
metrics = context_metrics.build_context_delay_metrics(
    buckets,
    line_ref=LINE_REF,
    direction_ref=DIRECTION_REF,
    day_type=DAY_TYPE,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
metrics


line_ref,line_name,direction_ref,hour_local,day_type,bucket_count,raw_poll_count,signed_mean_delay_min,median_delay_min,p75_delay_min,p90_delay_min,p95_delay_min,pct_over_3_min_late,pct_over_5_min_late,pct_early,pct_over_1_min_early,pct_over_3_min_early,median_early_min_abs,p90_early_min_abs
str,str,str,str,str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""24""","""24""","""1""","""07:00""","""weekend""",167,267,2.04,-1.68,0.56,28.39,29.6,12.57,12.57,70.66,56.89,24.55,2.53,4.83
"""402""","""402""","""2""","""22:00""","""weekday""",216,423,4.54,-1.1,0.14,27.83,27.83,20.37,20.37,65.74,55.56,9.72,1.6,3.43
"""24""","""24""","""1""","""17:00""","""weekend""",181,309,8.53,6.15,12.17,24.4,25.17,68.51,58.01,19.89,12.71,7.73,1.88,4.08
"""24""","""24""","""2""","""17:00""","""weekend""",178,339,4.57,1.03,4.82,21.4,21.89,40.45,24.72,33.71,16.85,2.25,0.98,2.39
"""25A""","""25A""","""1""","""10:00""","""weekday""",433,588,3.09,0.55,2.72,21.2,26.12,23.09,14.09,43.19,28.64,1.85,1.42,2.61
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""24""","""24""","""1""","""20:00""","""weekday""",493,869,6.49,6.0,9.77,11.26,15.24,84.58,59.23,6.29,4.87,0.61,1.71,2.99
"""24""","""24""","""2""","""14:00""","""weekday""",478,929,4.83,3.7,6.58,11.24,21.33,60.46,35.56,17.57,9.83,5.44,1.2,4.22
"""614""","""614""","""2""","""08:00""","""weekday""",621,1346,6.38,6.02,8.37,11.22,12.37,85.99,63.45,0.32,0.16,0.0,1.91,2.76


In [3]:
if metrics.is_empty():
    print("No matching observations found.")
else:
    chart_data = metrics.with_columns(
        (
            pl.col("line_ref")
            + pl.lit(" dir ")
            + pl.col("direction_ref").fill_null("?")
            + pl.lit(" ")
            + pl.col("hour_local")
            + pl.lit(" ")
            + pl.col("day_type")
        ).alias("context")
    )
    fig = px.bar(
        chart_data.sort("p90_delay_min"),
        x="p90_delay_min",
        y="context",
        orientation="h",
        title="Highest-delay line/hour contexts",
        labels={"context": "Context", "p90_delay_min": "P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
